Vectorización

In [65]:
import numpy as np
import pandas as pd
from scipy import sparse

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import normalize
from sklearn.utils.validation import check_is_fitted
from sentence_transformers import SentenceTransformer

In [66]:
def _to_text(value, use_pos=False):
    """Convierte texto o una lista de tokens (con o sin POS tags) en una cadena."""
    if value is None:
        return ""
    if isinstance(value, float) and np.isnan(value):
        return ""
    if isinstance(value, (list, tuple, np.ndarray)):
        tokens = []
        for t in value:
            if t is None:
                continue
            # Si el token viene como tupla/lista (palabra, tag_pos)
            if isinstance(t, (list, tuple)) and len(t) >= 2:
                word, pos = str(t[0]), str(t[1])
                if use_pos:
                    tokens.append(f"{word}_{pos}")
                else:
                    tokens.append(word)
            else:
                # Si es un string plano (u otro formato)
                tokens.append(str(t))
        return " ".join(tokens)
    return str(value)


class NLITextVectorizer(BaseEstimator, TransformerMixin):
    """
    Transformador personalizado compatible con scikit-learn para tareas de Inferencia 
    en Lenguaje Natural (NLI), diseñado para procesar columnas de texto o tokens 
    (con o sin etiquetas POS) y convertirlas en matrices de características numéricas 
    o vectores densos.

    Attributes:
        vectorizer_ (CountVectorizer o TfidfVectorizer): Ajustado según el método elegido.
        word_vectorizer_ (TfidfVectorizer): Usado en 'word_char' para palabras.
        char_vectorizer_ (TfidfVectorizer): Usado en 'word_char' para caracteres.
        svd_components_ (TruncatedSVD): Usado en 'tfidf_svd' para reducción dimensional.
        embedding_model_ (SentenceTransformer): Usado en 'embeddings'.
        n_features_in_ (int): Número de columnas de entrada procesadas.
    """

    VALID_METHODS = {
        "count", "binary", "relative", "tfidf",
        "tfidf_bigram", "tfidf_trigram", "char_tfidf",
        "word_char", "pair_separate", "pair_diff",
        "tfidf_svd", "embeddings"
    }

    def __init__(
        self,
        columns=("tokens",),
        method="tfidf",
        max_features=30000,
        min_df=1,
        word_ngram_range=(1, 1),
        char_ngram_range=(3, 5),
        analyzer="word",
        svd_components=300,
        normalize_embeddings=True,
        embedding_model="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
        embedding_batch_size=32,
        sublinear_tf=True,
        random_state=42,
        use_pos=False
    ):
        """
        Inicializa el vectorizador NLI.

        Args:
            columns (tuple o list of str): Nombres de las columnas de tokens del DataFrame de entrada 
                a procesar (por ejemplo, `("tokens",)` o `("premise", "hypothesis")`).
            method (str): Estrategia de vectorización a emplear. Opciones disponibles en `VALID_METHODS`:
                
                - **Enfoques de Recuento y TF-IDF clásicos:**
                  * `"count"`: Matriz dispersa con frecuencias absolutas de los términos.
                  * `"binary"`: Matriz dispersa binaria (1 si el término aparece, 0 si no).
                  * `"relative"`: Frecuencias relativas normalizadas (norma L1).
                  * `"tfidf"`: Ponderación clásica Term Frequency - Inverse Document Frequency.
                  * `"tfidf_bigram"`: TF-IDF analizando unigramas y bigramas (1, 2).
                  * `"tfidf_trigram"`: TF-IDF analizando unigramas hasta trigramas (1, 3).
                  * `"char_tfidf"`: TF-IDF basado en n-gramas de caracteres.
                  * `"word_char"`: Concatena horizontalmente(hstack) las características TF-IDF a nivel de palabras y de caracteres.

                - **Enfoques para Pares de Textos (NLI):**
                  * `"pair_separate"`: Vectoriza de forma independiente dos columnas (premise y hypothesis) 
                    y concatena horizontalmente sus matrices `[Xp, Xh]`.
                  * `"pair_diff"`: Combina premisa e hipótesis añadiendo la diferencia absoluta (`|Xp - Xh|`) 
                    y métricas de solapamiento semántico (Jaccard, coberturas y similitud de coseno).

                - **Reducción de Dimensionalidad y Modelos Profundos:**
                  * `"tfidf_svd"`: Aplica TF-IDF y reduce la dimensionalidad mediante Truncated SVD 
                    (Latent Semantic Analysis) generando una matriz densa.
                  * `"embeddings"`: Utiliza un modelo de Sentence Transformers para generar vectores densos 
                    semánticos. Si hay dos columnas, devuelve la concatenación: `[ep, eh, |ep - eh|, ep * eh]`.

            max_features (int): Número máximo de términos en el vocabulario (ordena por frecuencia). 
                Ayuda a controlar el consumo de memoria.
            min_df (int o float): Frecuencia mínima de documento. Descarta términos que aparezcan 
                en menos documentos que este valor.
            word_ngram_range (tuple): Rango `(min_n, max_n)` para n-gramas de palabras.
            char_ngram_range (tuple): Rango `(min_n, max_n)` para n-gramas de caracteres.
            analyzer (str): Nivel de análisis para el vectorizador base (`"word"` o `"char"`).
            svd_components (int): Número de componentes de salida al usar `tfidf_svd`.
            normalize_embeddings (bool): Si es True, normaliza los vectores de embeddings a norma L2.
            embedding_model (str): Identificador del modelo preentrenado de Hugging Face para SentenceTransformers.
            embedding_batch_size (int): Tamaño del lote para la generación de embeddings neuronales.
            sublinear_tf (bool): Aplica escala logarítmica a la frecuencia de término `1 + log(tf)`.
            random_state (int): Semilla aleatoria para la reproducibilidad en Truncated SVD.
            use_pos (bool): Si es True y los tokens vienen estructurados como tuplas `(palabra, pos_tag)`, 
                combina ambos como `"palabra_POS"` para tratarlos como entidades independientes. 
                Si es False, omite la etiqueta POS y usa únicamente la palabra.
        """
        self.columns = columns
        self.method = method
        self.max_features = max_features
        self.min_df = min_df
        self.word_ngram_range = word_ngram_range
        self.char_ngram_range = char_ngram_range
        self.analyzer = analyzer
        self.svd_components = svd_components
        self.normalize_embeddings = normalize_embeddings
        self.embedding_model = embedding_model
        self.embedding_batch_size = embedding_batch_size
        self.sublinear_tf = sublinear_tf
        self.random_state = random_state
        self.use_pos = use_pos

    def _get_texts(self, X):
        if not isinstance(X, pd.DataFrame):
            raise TypeError("X debe ser un DataFrame.")

        missing = [c for c in self.columns if c not in X.columns]
        if missing:
            raise ValueError(f"Faltan columnas: {missing}")
        
        return [
            pd.Series([_to_text(v, use_pos=self.use_pos) for v in X[c]]).reset_index(drop=True)
            for c in self.columns
        ]

    def _fit_corpus(self, vectorizer, texts):
        corpus = pd.concat(texts, ignore_index=True)
        vectorizer.fit(corpus)
        return vectorizer

    def _make_tfidf(self, analyzer="word", ngrams=None):
        return TfidfVectorizer(
            analyzer=analyzer,
            ngram_range=ngrams or self.word_ngram_range,
            min_df=self.min_df,
            max_features=self.max_features,
            sublinear_tf=self.sublinear_tf,
            dtype=np.float32,
        )

    def fit(self, X, y=None):
        if self.method not in self.VALID_METHODS:
            raise ValueError(f"Método no reconocido: {self.method}")

        texts = self._get_texts(X)
        self.n_features_in_ = len(self.columns)

        if self.method in {"pair_separate", "pair_diff"} and len(texts) != 2:
            raise ValueError(f"{self.method} necesita dos columnas.")

        # Vertical stack para tokens de premisa e hipótesis 
        # (len (corpus) = 2 * len(X)) si hay 2 columnas (premisa + hipotesis)
        corpus = pd.concat(texts, ignore_index=True)

        if self.method in {"count", "binary", "relative"}:
            self.vectorizer_ = CountVectorizer(
                binary=self.method == "binary",
                ngram_range=self.word_ngram_range,
                min_df=self.min_df,
                max_features=self.max_features,
                dtype=np.float32,
            )
            self.vectorizer_.fit(corpus)

        elif self.method == "tfidf":
            self.vectorizer_ = self._make_tfidf()
            self.vectorizer_.fit(corpus)

        elif self.method == "tfidf_bigram":
            self.vectorizer_ = self._make_tfidf(ngrams=(1, 2))
            self.vectorizer_.fit(corpus)

        elif self.method == "tfidf_trigram":
            self.vectorizer_ = self._make_tfidf(ngrams=(1, 3))
            self.vectorizer_.fit(corpus)

        elif self.method == "char_tfidf":
            self.vectorizer_ = self._make_tfidf(
                analyzer="char",
                ngrams=self.char_ngram_range
            )
            self.vectorizer_.fit(corpus)

        elif self.method == "word_char":
            self.word_vectorizer_ = self._make_tfidf()
            self.char_vectorizer_ = self._make_tfidf(
                analyzer="char",
                ngrams=self.char_ngram_range
            )
            self.word_vectorizer_.fit(corpus)
            self.char_vectorizer_.fit(corpus)

        elif self.method in {"pair_separate", "pair_diff"}:
            self.vectorizer_ = self._make_tfidf()
            self.vectorizer_.fit(corpus)

        elif self.method == "tfidf_svd":
            self.vectorizer_ = self._make_tfidf()
            self.vectorizer_.fit(corpus)

            matrix = self.vectorizer_.transform(corpus)
            max_valid = min(matrix.shape[0] - 1, matrix.shape[1] - 1)

            if max_valid < 1:
                raise ValueError("No hay suficientes datos para aplicar SVD.")

            self.svd_ = TruncatedSVD(
                n_components=min(self.svd_components, max_valid),
                random_state=self.random_state
            )
            self.svd_.fit(matrix)

        elif self.method == "embeddings":
            self.embedding_model_ = SentenceTransformer(
                self.embedding_model
            )

        return self

    def _encode(self, texts):
        return self.embedding_model_.encode(
            list(texts),
            batch_size=self.embedding_batch_size,
            convert_to_numpy=True,
            normalize_embeddings=self.normalize_embeddings,
            show_progress_bar=False,
        ).astype(np.float32)

    def _transform_one(self, texts, vectorizer):
        matrix = vectorizer.transform(texts)

        if self.method == "relative":
            return normalize(matrix, norm="l1")

        return matrix.tocsr()

    def _overlap_features(self, p, h, Xp, Xh):
        rows = []

        for a, b in zip(p, h):
            ta = set(a.split())
            tb = set(b.split())

            intersection = ta & tb
            union = ta | tb

            jaccard = len(intersection) / len(union) if union else 0
            coverage_p = len(intersection) / len(ta) if ta else 0
            coverage_h = len(intersection) / len(tb) if tb else 0

            rows.append([jaccard, coverage_p, coverage_h])

        overlap = sparse.csr_matrix(
            np.asarray(rows, dtype=np.float32)
        )

        numerator = np.asarray(
            Xp.multiply(Xh).sum(axis=1)
        ).ravel()

        norm_p = np.sqrt(
            np.asarray(Xp.multiply(Xp).sum(axis=1)).ravel()
        )
        norm_h = np.sqrt(
            np.asarray(Xh.multiply(Xh).sum(axis=1)).ravel()
        )

        denominator = norm_p * norm_h
        cosine = np.zeros_like(numerator)

        valid = denominator > 0
        cosine[valid] = numerator[valid] / denominator[valid]

        cosine = sparse.csr_matrix(cosine.reshape(-1, 1))

        return sparse.hstack(
            [overlap, cosine],
            format="csr"
        )

    def transform(self, X):
        check_is_fitted(self, "n_features_in_")
        texts = self._get_texts(X)

        if self.method in {
            "count", "binary", "relative", "tfidf",
            "tfidf_bigram", "tfidf_trigram", "char_tfidf"
        }:
            matrices = [
                self._transform_one(t, self.vectorizer_)
                for t in texts
            ]

            if len(matrices) == 1:
                return matrices[0]

            return sparse.hstack(matrices, format="csr")

        if self.method == "word_char":
            blocks = []

            for t in texts:
                word = self.word_vectorizer_.transform(t)
                char = self.char_vectorizer_.transform(t)
                blocks.extend([word, char])

            return sparse.hstack(blocks, format="csr")

        if self.method in {"pair_separate", "pair_diff"}:
            p, h = texts

            Xp = self.vectorizer_.transform(p).tocsr()
            Xh = self.vectorizer_.transform(h).tocsr()

            blocks = [Xp, Xh]

            if self.method == "pair_diff":
                blocks.append(abs(Xp - Xh))
                blocks.append(
                    self._overlap_features(p, h, Xp, Xh)
                )

            return sparse.hstack(blocks, format="csr")

        if self.method == "tfidf_svd":
            matrices = [
                self.svd_.transform(
                    self.vectorizer_.transform(t)
                )
                for t in texts
            ]
            return np.hstack(matrices).astype(np.float32)

        if self.method == "embeddings":
            embeddings = [self._encode(t) for t in texts]

            if len(embeddings) == 1:
                return embeddings[0]

            ep, eh = embeddings

            return np.hstack([
                ep,
                eh,
                np.abs(ep - eh),
                ep * eh
            ]).astype(np.float32)

        raise ValueError("Método no implementado.")

Ejemplos de vectorización

In [67]:
df_train = pd.DataFrame({
    "premise": [
        ["el", "gato", "duerme", "en", "el", "sofa"],
        ["un", "perro", "corre", "por", "el", "parque"]
    ],
    "hypothesis": [
        ["hay", "un", "felino", "descansando"],
        ["el", "animal", "esta", "quieto"]
    ]
})


vectorizer = NLITextVectorizer(
    columns=("premise", "hypothesis"),
    method="tfidf",
    use_pos=False, # No uso etiquetas POS, solo tengo tokens
    sublinear_tf=True, # Aplico escala logarítmica de la frecuencia
    max_features=5000,
    word_ngram_range=(1, 1), # Analizo para palabras unigramas solo 
    min_df=1  # Descarto términos que aparecen en menos de 1 documento
              # Numero de documentos = numero de instancias (si partimos de 1 columna de tokens)
              # o el doble de instancias si partimos de 2 columnas (premise, hypothesis), ver fit()
)

X_train_vectors = vectorizer.fit_transform(df_train)

print("Tipo de salida:", type(X_train_vectors))
print("Forma (shape) de la matriz de vectores:", X_train_vectors.shape)

X_train_vectors.toarray()


# Shape (2,32) debido a la concatenacion horizontal de las matrices de TF-IDF de premisa e hipótesis,
# cada una con 16 características (o palabras únicas) en este ejemplo.
# Matriz de Premisas: 2 filas × 16 columnas
# Matriz de Hipótesis: 2 filas × 16 columnas
# Concatenación horizontal (16 + 16): 2 filas × 32 columna
# Ver método transform() de NLITextVectorizer

Tipo de salida: <class 'scipy.sparse._csr.csr_matrix'>
Forma (shape) de la matriz de vectores: (2, 32)


array([[0.        , 0.        , 0.        , 0.43988705, 0.47539207,
        0.43988705, 0.        , 0.        , 0.43988705, 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.43988705,
        0.        , 0.        , 0.        , 0.52547276, 0.        ,
        0.        , 0.        , 0.        , 0.52547276, 0.        ,
        0.52547276, 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.41428876],
       [0.        , 0.44592217, 0.        , 0.        , 0.28462633,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.44592217, 0.44592217, 0.44592217, 0.        , 0.        ,
        0.35157016, 0.5417361 , 0.        , 0.        , 0.        ,
        0.34578314, 0.        , 0.5417361 , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.5417361 ,
        0.        , 0.        ]], dtype=float32)

In [68]:
vectorizer = NLITextVectorizer(
    columns=("premise", "hypothesis"),
    method="word_char",
    use_pos=False, # No uso etiquetas POS, solo tengo tokens
    sublinear_tf=True, # Aplico escala logarítmica de la frecuencia
    max_features=5000,
    word_ngram_range=(1, 1), # Analizo unigramas solo para palabras
    char_ngram_range=(3, 5), # Analizo n-gramas de caracteres de longitud 3 a 5
    min_df=1  # Descarto términos que aparecen en menos de 1 documento
              # Numero de documentos = numero de instancias (si partimos de 1 columna de tokens)
              # o el doble de instancias si partimos de 2 columnas (premise, hypothesis), ver fit()
)

X_train_vectors = vectorizer.fit_transform(df_train)

print("Tipo de salida:", type(X_train_vectors))
print("Forma (shape) de la matriz de vectores:", X_train_vectors.shape)

X_train_vectors.toarray()

# A nivel de palabras (word): Extrae los unigramas de palabras. 
# Hay 16 palabras únicas en el corpus, este bloque aporta 16 columnas para premisa y 16 para hipotesis

#A nivel de caracteres (char): Extrae n-gramas de caracteres con el rango (3, 5) (secuencias de 3, 4 y 5 letras, incluyendo espacios y sufijos/raíces). 
# Al generar todas las combinaciones posibles de caracteres este vectorizador encuentra 254 características únicas en la premisa y 254 características únicas en la hipótesis.

# Total (al combinar horizontalmente): 16 + 16 + 254 + 254 = 540 columnas (características) en la matriz final de vectores.

Tipo de salida: <class 'scipy.sparse._csr.csr_matrix'>
Forma (shape) de la matriz de vectores: (2, 540)


array([[0.        , 0.        , 0.        , ..., 0.12380336, 0.12380336,
        0.12380336],
       [0.        , 0.44592217, 0.        , ..., 0.        , 0.        ,
        0.        ]], shape=(2, 540), dtype=float32)

In [69]:
df_1col_pos = pd.DataFrame({
    "tokens": [
        [("el", "DET"), ("gato", "NOUN"), ("duerme", "VERB"), ("entonces", "ADV"), ("hay", "VERB"), ("un", "DET"), ("felino", "NOUN")],
        [("un", "DET"), ("perro", "NOUN"), ("corre", "VERB"), ("por", "ADP"), ("el", "DET"), ("parque", "NOUN")]
    ]
})

vectorizer_1 = NLITextVectorizer(
    columns=("tokens",),
    method="embeddings",
    use_pos=True  # Activo POS tags
)

X_vecs_1 = vectorizer_1.fit_transform(df_1col_pos)

print("Forma (1 col, con POS):", X_vecs_1.shape) 
# Salida: (2, 384) -> Una sola matriz de embeddings de 384 dimensiones por instancia.
X_vecs_1 # No hace falta llamar a toarray() ya que la salida es un array de numpy, 
         # en vez de una matriz dispersa como en los tfidf o count vectorizer.

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 867.98it/s]


Forma (1 col, con POS): (2, 384)


array([[ 9.48386919e-03,  5.48957773e-02,  1.94618832e-02,
         4.92168926e-02, -3.01706679e-02, -5.06459028e-02,
         1.62036791e-01,  4.48887572e-02,  5.04726283e-02,
         6.83170417e-03,  7.50034079e-02, -1.18865639e-01,
         3.39856334e-02,  3.62386629e-02,  2.46663541e-02,
        -3.11030969e-02, -5.99834509e-03,  3.88443545e-02,
         2.06991341e-02, -4.23391536e-02,  1.57786593e-01,
         6.71886490e-04, -4.67847474e-03, -5.67039102e-03,
        -5.09243235e-02, -4.21363600e-02, -1.89830363e-02,
         4.33225445e-02,  1.14803491e-02, -2.53871847e-02,
        -6.32880535e-03,  2.64084134e-02,  2.23178566e-02,
         6.49632737e-02,  2.28538401e-02,  1.43103302e-02,
         3.85676557e-03, -5.22393361e-02, -1.01761185e-01,
        -5.16051380e-03, -6.17914945e-02, -2.31401864e-02,
        -4.38246429e-02,  2.04741620e-02,  2.99585648e-02,
         1.65384787e-03, -4.11018245e-02, -1.06257498e-02,
        -3.00444700e-02,  1.67709216e-02,  6.48535602e-0

In [70]:
df_2col_pos = pd.DataFrame({
    "premise": [
        [("el", "DET"), ("gato", "NOUN"), ("duerme", "VERB")],
        [("un", "DET"), ("perro", "NOUN"), ("corre", "VERB")]
    ],
    "hypothesis": [
        [("hay", "VERB"), ("un", "DET"), ("felino", "NOUN")],
        [("el", "animal", "NOUN"), ("esta", "AUX"), ("quieto", "ADJ")]
    ]
})

vectorizer_2 = NLITextVectorizer(
    columns=("premise", "hypothesis"),
    method="embeddings",
    use_pos=True
)

X_vecs_2 = vectorizer_2.fit_transform(df_2col_pos)

print("Forma (2 cols, con POS):", X_vecs_2.shape) 
# Aquí la salida es (2, 1536) porque el vectorizador genera embeddings de 384 dimensiones para cada columna (premise y hypothesis)
# Luego concatena horizontalmente los vectores de premisa, hipótesis, la diferencia absoluta y el producto elemento a elemento (muy poderoso para transformers)
# Resultado: 4 * 384 = 1536 características por instancia. 
# Salida: (2, 1536) -> 384 (premise) + 384 (hypothesis) + 384 (diferencia abs) + 384 (producto)
X_vecs_2

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 875.02it/s]


Forma (2 cols, con POS): (2, 1536)


array([[-1.3074461e-02,  3.3406179e-02, -5.9471643e-03, ...,
        -5.5743301e-05,  2.4317260e-04, -2.5384803e-04],
       [-3.7877338e-03,  4.9693786e-02, -4.9065221e-02, ...,
         3.3497654e-05,  3.7586762e-04,  5.4976642e-03]],
      shape=(2, 1536), dtype=float32)

In [71]:
df_1col_nopos = pd.DataFrame({
    "texto_completo": [
        [("el", "DET"), ("gato", "NOUN"), ("duerme", "VERB")],
        [("un", "DET"), ("perro", "NOUN"), ("corre", "VERB")]
    ]
})

vectorizer_3 = NLITextVectorizer(
    columns=("texto_completo",),
    method="embeddings",
    use_pos=False  # Ignora las etiquetas POS (mismo resultado que si se pasara una lista de strings sin tuplas)
                   # Se vectoriza del mismo modo 2 tokens idénticos pero con distinta categoría gramatical (ej: hunt verb y hunt noun)
)


X_vecs_3 = vectorizer_3.fit_transform(df_1col_nopos)

print("Forma (1 col, sin POS):", X_vecs_3.shape) 
# Salida: (2, 384) -> Devuelve únicamente el embedding limpio del texto completo.

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 853.42it/s]


Forma (1 col, sin POS): (2, 384)


In [72]:
df_2col_nopos = pd.DataFrame({
    "premise": [
        ["el", "gato", "duerme", "en", "el", "sofa"],
        ["un", "perro", "corre", "por", "el", "parque"]
    ],
    "hypothesis": [
        ["hay", "un", "felino", "descansando"],
        ["el", "animal", "esta", "quieto"]
    ]
})

vectorizer_4 = NLITextVectorizer(
    columns=("premise", "hypothesis"),
    method="embeddings",
    use_pos=False
)

X_vecs_4 = vectorizer_4.fit_transform(df_2col_nopos)

print("Forma (2 cols, sin POS):", X_vecs_4.shape) 
# Salida: (2, 1536) -> Concatena [premisa, hipótesis, |premisa - hipótesis|, premisa * hipótesis]
X_vecs_4

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 850.43it/s]


Forma (2 cols, sin POS): (2, 1536)


array([[ 0.09331264, -0.05714897, -0.02595772, ...,  0.00011908,
         0.00151414, -0.00072391],
       [ 0.08655477,  0.02148705, -0.09319922, ...,  0.00048628,
         0.00056532,  0.00189124]], shape=(2, 1536), dtype=float32)

In [73]:
df_nli = pd.DataFrame({
    "premise": [["el", "gato", "duerme", "en", "el", "sofa"]],
    "hypothesis": [["hay", "un", "felino", "descansando"]]
})

# pair_diff (este metodo solo se admite separando premisa y conclusion)
vectorizer = NLITextVectorizer(
    columns=("premise", "hypothesis"),
    method="pair_diff",
    word_ngram_range=(1, 1),
    max_features=5000,
    min_df=1,
    use_pos=False
)

X_vecs = vectorizer.fit_transform(df_nli)
print("Forma con pair_diff:", X_vecs.shape)
# Salida: (1, Vocabulario_Premisa + Vocabulario_Hipotesis + Vocabulario_Diferencia + 4_features_de_solapamiento)
X_vecs.toarray()

Forma con pair_diff: (1, 31)


array([[0.        , 0.38161415, 0.64612895, 0.38161415, 0.        ,
        0.38161415, 0.        , 0.38161415, 0.        , 0.5       ,
        0.        , 0.        , 0.        , 0.5       , 0.        ,
        0.5       , 0.        , 0.5       , 0.5       , 0.38161415,
        0.64612895, 0.38161415, 0.5       , 0.38161415, 0.5       ,
        0.38161415, 0.5       , 0.        , 0.        , 0.        ,
        0.        ]], dtype=float32)

In [74]:
df_svd = pd.DataFrame({
    "premise": [
        ["el", "gato", "duerme", "en", "el", "sofa"],
        ["un", "perro", "corre", "por", "el", "parque"]
    ],
    "hypothesis": [
        ["hay", "un", "felino", "descansando"],
        ["el", "animal", "esta", "quieto"]
    ]
})

vectorizer = NLITextVectorizer(
    columns=("premise", "hypothesis"),
    method="tfidf_svd",
    word_ngram_range=(1, 1),
    max_features=5000,
    min_df=1,
    svd_components=50  # Reducimos a 50 dimensiones por columna (2 columnas en total)
                       # Evito vectores enormes como tfidf
)

X_vecs = vectorizer.fit_transform(df_svd)

print("Tipo de salida:", type(X_vecs))  # numpy.ndarray (densa)
print("Forma con tfidf_svd:", X_vecs.shape)

X_vecs 

# Salida obtenida: (2, 6)

# Debería salir idealmente: (2, 100)  (50 dimensiones del vector premisa + 50 del vector hipótesis) 
# El motivo por el cual sale (2, 6) es debido al tamaño reducido de nuestro DataFrame de prueba:
# max_valid = min(matrix.shape[0] - 1, matrix.shape[1] - 1)
# 
# 1. El tamaño del corpus (matrix.shape[0]) es 4: 
#    - Se concatenan verticalmente las 2 filas de "premise" y las 2 de "hypothesis" en el fit().
#    - Por tanto, el número de documentos es 4, lo que nos da (4 - 1) = 3 como límite por filas.
#
# 2. El tamaño del vocabulario (matrix.shape[1]) es el número de palabras únicas del corpus global.
#    - Si el vocabulario global tuviera, por ejemplo, 16 palabras únicas, el límite por columnas sería (16 - 1) = 15.
#
# 3. Al hacer el min() global: max_valid = min(3 (por filas), 15 (por vocabulario)) = 3.
#
# 4. Finalmente, el SVD calcula: min(self.svd_components, max_valid) -> min(50, 3) = 3.
#    - Como el resultado es 3, el SVD solo puede generar un máximo de 3 componentes válidos por columna.
#    - Al tener 2 columnas ("premise" e "hypothesis"), se concatenan horizontalmente: 3 + 3 = 6 columnas en total.
#    - Como el DataFrame tiene 2 filas (instancias), la forma final resultante es (2, 6).

Tipo de salida: <class 'numpy.ndarray'>
Forma con tfidf_svd: (2, 6)


array([[ 0.6370161 , -0.3733249 , -0.4556326 ,  0.33051196,  0.7880738 ,
         0.2879901 ],
       [ 0.6596611 ,  0.34528226, -0.30540177,  0.58354294, -0.4291427 ,
         0.67950964]], dtype=float32)

In [75]:
df_svd = pd.DataFrame({
    "tokens": [
        ["el", "gato", "duerme", "en", "el", "sofa", "hay", "un", "felino", "descansando"],
         ["un", "perro", "corre", "por", "el", "parque", "el", "animal", "esta", "quieto"]
    ]
})

vectorizer = NLITextVectorizer(
    columns=("tokens", ),
    method="tfidf_svd",
    word_ngram_range=(1, 1),
    max_features=5000,
    min_df=1,
    svd_components=50  # Reducimos a 50 dimensiones por columna (en este caso 1 columna)
                       # Evito vectores enormes como tfidf
)

X_vecs = vectorizer.fit_transform(df_svd)

print("Tipo de salida:", type(X_vecs))  # numpy.ndarray (densa)
print("Forma con tfidf_svd:", X_vecs.shape) 

# Salida: (2, 1)

# No sale (2, 50) (ver explicacion de la celda anterior)
X_vecs

Tipo de salida: <class 'numpy.ndarray'>
Forma con tfidf_svd: (2, 1)


array([[0.7805552 ],
       [0.78055555]], dtype=float32)